# AAA Project Two

This project focuses on feature transforming. After data exploration, we will pick a target variable to predict, and identify/transform features for our Model.

Remeber this is a continuous imporvement process. You may need to come back to refine your features after you start to build your machine learning model.

Objective: You will select a target variable to predict and identify and transform features for your model. Our data will be transformed at the member level, meaning each member will have one row. As an example, I have generated features based on trip cancellations, depart month, travel channel, and some demographic information for each member. Please develop additional features from the travel booking table and other tables that would be useful for model building. Please add additional featurs you have transformed at the end of this notebook. You need to develop features based on at least five other columns. Please join all features and save features to your directory.

Due date: Tuesday 6/17.

# Import Functions

In [ ]:
# append path to utilities to system path
import sys
#sys.path.append('/opt/shared/')

from pyspark.sql import SparkSession
import numpy as np
from wordcloud import WordCloud
from PIL import Image
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import udf, desc, col, isnull, to_timestamp, to_date, trim, year, month, when, sum, lit, col, lag, datediff, upper
from pyspark.sql.types import StringType, IntegerType, DoubleType, ArrayType, MapType, LongType, BooleanType
from pyspark.sql.window import Window
from pyspark.ml.feature import Imputer
from pyspark.ml.functions import vector_to_array
import utilities as U
import warnings

# Ignore all warnings
warnings.filterwarnings('ignore')

# change the account name to your email account
account='sli'

# define a data path to store the data 
data_path='/net/clusterhn/home/'+account+'/msds630/aaa_data/'

# Start a Spark Session

In [ ]:
# check if the Spark session is active. If it is activate, close it
try:
    if spark:
        spark.stop()
except:
    pass    

spark = (SparkSession.builder.appName("AAA Travelling Data Analysis and Modeling Buidling")
        .config("spark.port.maxRetries", "200")
        .config("spark.sql.mapKeyDedupPolicy", "LAST_WIN")  # This configuration allow the duplicate keys in the map data type.
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")
        .getOrCreate())

# confiture the log level (defaulty is WWARN)
spark.sparkContext.setLogLevel('ERROR')

# Set Spark configuration for handling datetime compatibility
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")

In [ ]:
# read all original files
# travel_booking=spark.read.parquet('/opt/shared/msds630_summer2025/aaa_data/parquet/20241024_trv_booking_summary.parquet').drop('_c0', 'id')
# benefits=spark.read.parquet('/opt/shared/msds630_summer2025/aaa_data/parquet/2024103_benefits.parquet')
# ers=spark.read.parquet('/opt/shared//msds630_summer2025/aaa_data/parquet/2024103_ers.parquet')
# insurance_engagement=spark.read.parquet('/opt/shared//msds630_summer2025/aaa_data/parquet/2024103_insurance_engagement.parquet')
# travel_credit=spark.read.parquet('/opt/shared//msds630_summer2025/aaa_data/parquet/2024103_travel_credit.parquet')
# visa=spark.read.parquet('/opt/shared//msds630_summer2025/aaa_data/parquet/2024103_visa.parquet')

# Load the cleaned data that include travel booking and individual data

We will prepare feature for predicting cancellation. I will remove pandamic year (2020) since it does not represent normal cancellation. I will foucs on Cruiser cancellation.

In [ ]:
from pyspark.sql import SparkSession

# Stop Spark if it's already running
try:
    if spark:
        spark.stop()
except:
    pass

# Start a new Spark session
spark = (SparkSession.builder
         .appName("AAA Travelling Data Analysis and Modeling Building")
         .config("spark.port.maxRetries", "200")
         .config("spark.sql.mapKeyDedupPolicy", "LAST_WIN")
         .config("spark.driver.memory", "8g")
         .config("spark.executor.memory", "8g")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")


In [ ]:
# Load travel booking data (main file to predict cancellation)
travel_booking = spark.read.parquet('/opt/shared/msds630_summer2025/aaa_data/parquet/20241024_trv_booking_summary.parquet')

# Optional: load others as needed
benefits = spark.read.parquet('/opt/shared/msds630_summer2025/aaa_data/parquet/2024103_benefits.parquet')
ers = spark.read.parquet('/opt/shared/msds630_summer2025/aaa_data/parquet/2024103_ers.parquet')
insurance = spark.read.parquet('/opt/shared/msds630_summer2025/aaa_data/parquet/2024103_insurance_engagement.parquet')
credit = spark.read.parquet('/opt/shared/msds630_summer2025/aaa_data/parquet/2024103_travel_credit.parquet')
visa = spark.read.parquet('/opt/shared/msds630_summer2025/aaa_data/parquet/2024103_visa.parquet')


In [ ]:
numeric_cols = [field.name for field in travel_booking.schema.fields if str(field.dataType) in ['IntegerType', 'DoubleType', 'LongType']]
travel_booking.select(numeric_cols).describe().show()


In [ ]:
# change the account name to your email account
account='sli'

# define a data path to store the data 
data_path='/net/clusterhn/home/'+account+'/msds630/aaa_data/'

df=spark.read.parquet(data_path+'travel_book_summary_cleaned.parquet')

In [ ]:
print(travel_booking.columns)

In [ ]:
from pyspark.sql.functions import col, when, avg, count, sum, lag, to_date
from pyspark.sql.window import Window

# Step 1: Create binary cancel label
travel_booking = travel_booking.withColumn(
    "cancel_label", when(col("STATUS") == "cancelled", 1).otherwise(0)
)

# Step 2: Keep required columns (including ID!)
required_cols = ["Individual_ID", "tripValue", "bookingCount", "travelTypeDescription", "Join_AAA_Date", "cancel_label"]
travel_booking = travel_booking.dropna(subset=required_cols)

# Step 3: Cast bookDate to date (if needed)
travel_booking = travel_booking.withColumn("bookDate", to_date("bookDate"))

# Step 4: Rolling historical aggregations per member
win = Window.partitionBy("Individual_ID").orderBy("bookDate")
travel_booking = travel_booking \
    .withColumn("prev_tripValue", lag("tripValue", 1).over(win)) \
    .withColumn("avg_tripValue_so_far", avg("tripValue").over(win)) \
    .withColumn("trip_count_so_far", count("tripValue").over(win))

# Step 5: Drop any null rows in features (esp. first rows without lag)
travel_booking = travel_booking.dropna(subset=["prev_tripValue", "avg_tripValue_so_far"])

# Step 6: Aggregate to member level
from pyspark.sql.functions import when

member_features = travel_booking.groupBy("Individual_ID").agg(
    avg("tripValue").alias("avg_tripValue"),
    count("*").alias("total_bookings"),
    sum("cancel_label").alias("total_cancelled"),
    (sum("cancel_label") / count("*")).alias("cancel_rate")
).withColumn(
    "member_cancel_label", when(col("total_cancelled") > 0, 1).otherwise(0)
)


In [ ]:
from pyspark.sql.functions import when

# Create binary target column
travel_booking = travel_booking.withColumn(
    "insurance_label",
    when(col("Insurancepurchased") == "yes", 1).otherwise(0)
)

# Drop rows with missing important features
selected_cols = [
    "travelTypeDescription", "tripValue", "bookingCount",
    "age", "gender", "Education", "Credit", "Wealth",
    "channel", "HomeownerRenter", "PresenceOfChildren", "insurance_label"
]

travel_booking = travel_booking.dropna(subset=selected_cols)

# Optional preview
travel_booking.select("travelTypeDescription", "tripValue", "gender", "insurance_label").show(5)


In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# Step 1: Index categorical columns
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_index", handleInvalid="keep")
    for col in ["travelTypeDescription", "gender", "Education", "Credit", "Wealth", "channel", "HomeownerRenter", "PresenceOfChildren"]
]

# Step 2: OneHotEncode indexed columns
encoder = OneHotEncoder(
    inputCols=[f"{col}_index" for col in ["travelTypeDescription", "gender", "Education", "Credit", "Wealth", "channel", "HomeownerRenter", "PresenceOfChildren"]],
    outputCols=[f"{col}_ohe" for col in ["travelTypeDescription", "gender", "Education", "Credit", "Wealth", "channel", "HomeownerRenter", "PresenceOfChildren"]]
)

# Step 3: Assemble all features
assembler = VectorAssembler(
    inputCols=[f"{col}_ohe" for col in ["travelTypeDescription", "gender", "Education", "Credit", "Wealth", "channel", "HomeownerRenter", "PresenceOfChildren"]] + ["tripValue", "bookingCount", "age"],
    outputCol="features"
)

# Step 4: Create pipeline
pipeline = Pipeline(stages=indexers + [encoder, assembler])

# Step 5: Transform data
pipeline_model = pipeline.fit(travel_booking)
final_data = pipeline_model.transform(travel_booking)

# Preview
final_data.select("features", "insurance_label").show(5, truncate=False)


In [ ]:
from pyspark.sql.functions import trim, upper

# Standardize text
travel_booking = travel_booking.withColumn("Insurancepurchased", trim(upper(col("Insurancepurchased"))))

# Define what counts as "purchased"
insurance_yes_values = [
    "VENDOR", "ACCEPTED", "INSURANCE PURCHASED INDEPENDENTLY",
    "AAA TRIP PROTECTION", "AAA TRIP PROTECTION ON SEPARATE INVOICE",
    "AAA TRIP PROTECTION AND VENDOR"
]

# Create binary label
travel_booking = travel_booking.withColumn(
    "insurance_label",
    when(col("Insurancepurchased").isin(insurance_yes_values), 1).otherwise(0)
)


In [ ]:
travel_booking.groupBy("insurance_label").count().show()


In [ ]:
travel_booking.groupBy("Insurancepurchased").count().orderBy("count", ascending=False).show(100, truncate=False)



In [ ]:
from pyspark.sql.types import IntegerType

# Force cast to int
travel_booking = travel_booking.withColumn("insurance_label", col("insurance_label").cast(IntegerType()))


In [ ]:
predictions.select("rawPrediction", "probability", "prediction").show(5, truncate=False)


In [ ]:
final_data.select("insurance_label", "features").show(5, truncate=False)


In [ ]:
from pyspark.sql.functions import when

# Add a weight column
weighted_data = final_data.withColumn(
    "weight",
    when(col("insurance_label") == 1, 5.0).otherwise(1.0)
)


In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features", 
    labelCol="insurance_label", 
    weightCol="weight"
)

lr_model = lr.fit(weighted_data)
predictions = lr_model.transform(weighted_data)


In [ ]:
predictions.select("insurance_label", "rawPrediction", "probability", "prediction").show(5, truncate=False)


In [ ]:
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import when, col

# Recap: make sure label is int
travel_booking = travel_booking.withColumn("insurance_label", col("insurance_label").cast(IntegerType()))

# Add weight column for imbalance
final_data = final_data.withColumn("weight", when(col("insurance_label") == 1, 5.0).otherwise(1.0))


In [ ]:
from pyspark.ml.classification import LogisticRegression

# Split
train_data, test_data = final_data.randomSplit([0.7, 0.3], seed=42)

# Train with weightCol
lr = LogisticRegression(
    featuresCol="features",
    labelCol="insurance_label",
    weightCol="weight",
    maxIter=20,
    regParam=0.1,
    elasticNetParam=0.0
)

lr_model = lr.fit(train_data)
predictions = lr_model.transform(test_data)


In [ ]:
predictions.select("insurance_label", "rawPrediction", "probability", "prediction").show(5, truncate=False)


In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="insurance_label",
    rawPredictionCol="rawPrediction",  # this works if output is valid
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(predictions)
print(f"AUC = {auc:.4f}")


In [ ]:
from pyspark.ml.functions import vector_to_array
final_data = final_data.withColumn("features", col("features").cast("vector"))


In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Train
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="insurance_label",
    weightCol="weight",  # still use weight for imbalance
    numTrees=50,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(train_data)
rf_predictions = rf_model.transform(test_data)

# Evaluate
evaluator = BinaryClassificationEvaluator(labelCol="insurance_label", metricName="areaUnderROC")
auc = evaluator.evaluate(rf_predictions)
print(f"Random Forest AUC = {auc:.4f}")


In [ ]:
final_data.select("features").rdd.map(lambda row: row['features'].toArray().tolist()).take(5)


In [ ]:
train_data.groupBy("insurance_label").count().show()


In [ ]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="insurance_label",
    weightCol="weight",
    numTrees=10,
    maxDepth=5,
    seed=42
)

rf_model = rf.fit(final_data)
rf_predictions = rf_model.transform(final_data)

rf_predictions.select("insurance_label", "rawPrediction", "probability", "prediction").show(5, truncate=False)


In [ ]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import DoubleType

# UDF to extract class-1 probability from probability vector
get_prob = udf(lambda v: float(v[1]), DoubleType())

# Add score column
rf_scored = rf_predictions.withColumn("score", get_prob(col("probability")))



In [ ]:
# You’re allowed to use pandas briefly for eval inside notebook
df_auc = rf_scored.select("insurance_label", "score").toPandas()

from sklearn.metrics import roc_auc_score
auc = roc_auc_score(df_auc["insurance_label"], df_auc["score"])
print(f"Manual AUC = {auc:.4f}")


In [ ]:
from pyspark.sql.functions import avg, count, sum

# Aggregate features to the member level
member_features = travel_booking.groupBy("Individual_ID").agg(
    avg("tripValue").alias("avg_tripValue"),
    count("*").alias("total_bookings"),
    sum("cancel_label").alias("total_cancelled"),
    (sum("cancel_label") / count("*")).alias("cancel_rate")
).withColumn("member_cancel_label", when(col("total_cancelled") > 0, 1).otherwise(0))


In [ ]:
from pyspark.sql.functions import when

# Binary target: 1 if they ever cancelled, else 0
member_features = member_features.withColumn(
    "member_cancel_label",
    when(col("total_cancelled") > 0, 1).otherwise(0)
)


In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["avg_tripValue", "total_bookings", "cancel_rate"],
    outputCol="features"
)

final_data = assembler.transform(member_features).select("features", "member_cancel_label")


In [ ]:
from pyspark.ml.feature import VectorAssembler

# Use only numeric behavioral features
assembler = VectorAssembler(
    inputCols=["prev_tripValue", "avg_tripValue_so_far", "trip_count_so_far"],
    outputCol="features"
)

final_data = assembler.transform(travel_booking).select("features", "cancel_label")


In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# Split
train_data, test_data = final_data.randomSplit([0.7, 0.3], seed=42)

# Train
rf = RandomForestClassifier(labelCol="cancel_label", featuresCol="features", seed=42)
rf_model = rf.fit(train_data)
predictions = rf_model.transform(test_data)


In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col, when
from pyspark.sql.types import IntegerType

# 1. Vector Assembler (numeric features only)
assembler = VectorAssembler(
    inputCols=["prev_tripValue", "avg_tripValue_so_far", "trip_count_so_far"],
    outputCol="features"
)
final_data = assembler.transform(travel_booking).select("features", "cancel_label")

# 2. Cast label to int
final_data = final_data.withColumn("cancel_label", col("cancel_label").cast(IntegerType()))

# 3. Train-test split
train_data, test_data = final_data.randomSplit([0.7, 0.3], seed=42)

# 4. Train model
rf = RandomForestClassifier(labelCol="cancel_label", featuresCol="features", seed=42)
rf_model = rf.fit(train_data)
predictions = rf_model.transform(test_data)

# 5. Evaluate AUC using rawPrediction (not probability)
evaluator = BinaryClassificationEvaluator(
    labelCol="cancel_label",
    rawPredictionCol="rawPrediction",  # required for AUC with RF
    metricName="areaUnderROC"
)

# 6. Get AUC
auc = evaluator.evaluate(predictions)
print(f"AUC = {auc:.4f}")


In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

get_prob = udf(lambda v: float(v[1]), DoubleType())
predictions = predictions.withColumn("score", get_prob(col("probability")))

# Convert to pandas for sklearn eval
df_auc = predictions.select("cancel_label", "score").toPandas()

from sklearn.metrics import roc_auc_score
auc = roc_auc_score(df_auc["cancel_label"], df_auc["score"])
print(f"AUC = {auc:.4f}")


In [ ]:
from pyspark.sql.functions import ntile
from pyspark.sql.window import Window

# Create deciles based on score
w = Window.orderBy(col("score").desc())
predictions = predictions.withColumn("decile", ntile(10).over(w))


In [ ]:
predictions.groupBy("decile") \
    .agg(
        count("*").alias("total"),
        sum("cancel_label").alias("cancelled")
    ) \
    .withColumn("cancel_rate", col("cancelled") / col("total")) \
    .orderBy("decile") \
    .show()


## data cleaning


In [ ]:
from pyspark.sql.functions import col, when, to_date

# Create binary target variable: 1 if cancelled, 0 otherwise
travel_booking = travel_booking.withColumn(
    "cancel_label",
    when(col("STATUS") == "cancelled", 1).otherwise(0)
)

# Clean necessary fields
required_cols = ["tripValue", "bookingCount", "travelTypeDescription", "Join_AAA_Date", "cancel_label"]
travel_booking = travel_booking.dropna(subset=required_cols)

# Cast to date (if needed)
travel_booking = travel_booking.withColumn("bookDate", to_date(col("bookDate")))


## Project



In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, count, lag

# Define window by member, ordered by date
win = Window.partitionBy("Individual_ID").orderBy("bookDate")

# Add lag and aggregates
travel_booking = travel_booking \
    .withColumn("prev_tripValue", lag("tripValue", 1).over(win)) \
    .withColumn("avg_tripValue_so_far", avg("tripValue").over(win)) \
    .withColumn("trip_count_so_far", count("tripValue").over(win))

# Drop rows with nulls in new features
travel_booking = travel_booking.dropna(subset=["prev_tripValue", "avg_tripValue_so_far"])


In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["prev_tripValue", "avg_tripValue_so_far", "trip_count_so_far"],
    outputCol="features"
)

final_data = assembler.transform(travel_booking).select("features", "cancel_label")


In [ ]:
from pyspark.sql.types import IntegerType

final_data = final_data.withColumn("cancel_label", col("cancel_label").cast(IntegerType()))


In [ ]:
train_data, test_data = final_data.randomSplit([0.7, 0.3], seed=42)


In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(labelCol="cancel_label", featuresCol="features", seed=42)
rf_model = rf.fit(train_data)
predictions = rf_model.transform(test_data)


In [ ]:
# Check class distribution
print(df_auc["cancel_label"].value_counts())


In [ ]:
final_data.groupBy("cancel_label").count().show()


In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "prev_tripValue",
        "avg_tripValue_so_far",
        "trip_count_so_far",
        "bookingCount",         # new feature
        "age"                   # another one
    ],
    outputCol="features"
)

final_data = assembler.transform(travel_booking).select("features", "cancel_label")


In [ ]:
travel_booking = travel_booking.withColumn(
    "cancel_label",
    when(col("STATUS") == "cancelled", 1).otherwise(0)
)


In [ ]:
travel_booking.groupBy("STATUS").count().show()


In [ ]:
from pyspark.sql.functions import lower

travel_booking = travel_booking.withColumn(
    "cancel_label",
    when(lower(col("STATUS")) == "cancelled", 1).otherwise(0)
)


In [ ]:
final_data = assembler.transform(travel_booking).select("features", "cancel_label")
final_data.groupBy("cancel_label").count().show()


In [ ]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType
from sklearn.metrics import roc_auc_score

# Step 1: UDF to extract score from probability vector safely
get_score = udf(lambda v: float(v[1]) if v and len(v) > 1 else (float(v[0]) if v else 0.0), DoubleType())
predictions = predictions.withColumn("score", get_score(col("probability")))

# Step 2: Convert to pandas
df_auc = predictions.select("cancel_label", "score").toPandas()

# Step 3: Check class distribution
label_counts = df_auc["cancel_label"].value_counts()
print(label_counts)

# Step 4: Compute AUC only if both classes exist
if set(df_auc["cancel_label"].unique()) == {0, 1}:
    auc = roc_auc_score(df_auc["cancel_label"], df_auc["score"])
    print(f"AUC = {auc:.4f}")
else:
    print("❌ AUC cannot be computed — only one class present in the test set.")


In [ ]:
# Try multiple seeds to ensure test set has both 0 and 1
for i in range(10):
    train_data, test_data = final_data.randomSplit([0.7, 0.3], seed=42 + i)
    df_test = test_data.select("cancel_label").toPandas()
    if 0 in df_test["cancel_label"].values and 1 in df_test["cancel_label"].values:
        print(f"✅ Found good split on attempt {i+1}")
        break
else:
    raise ValueError("❌ Failed to find a test split with both 0 and 1 classes.")


In [ ]:
from pyspark.sql import Row
import numpy as np

# Convert vector column to separate numeric columns
features_rdd = final_data.select("features").rdd.map(lambda row: row['features'].toArray().tolist())

# Dynamically create schema names
num_features = final_data.select("features").first()["features"].size
feature_names = [f"f{i}" for i in range(num_features)]

# Convert to Spark DataFrame
features_df = features_rdd.map(lambda x: Row(**{f"f{i}": float(x[i]) for i in range(len(x))})).toDF()

# Describe statistics
features_df.describe().show()


## Previous trip
We want to see what trip an indiviudal booked before booking a cruiser

In [ ]:
df1=df.select('individual_id', 'book_date', 'travel_type_description', 'trip_value')
df1.show(10)

In [ ]:
windowSpec=Window.partitionBy('individual_id').orderBy('book_date')

df2=df1.withColumn('book_date_previous', lag('book_date').over(windowSpec)).\
        withColumn('travel_type_description_previous', lag('travel_type_description').over(windowSpec)).\
        withColumn('trip_value_previous', lag('trip_value').over(windowSpec)).filter(col('travel_type_description')=='Ship').filter(col('book_date_previous').isNotNull()).\
        withColumn('days_before_cruiser', datediff('book_date', 'book_date_previous'))
        

df2.show()

In [ ]:
df2.groupBy('travel_type_description_previous').count().orderBy(desc('count')).show()

In [ ]:
df2.groupBy('travel_type_description_previous') \
   .count() \
   .agg(sum('count').alias('total_count')) \
   .show()


In [ ]:
df2.select('days_before_cruiser').summary().show()

In [ ]:
df2.groupBy('individual_id', 'travel_type_description_previous').count().show()

In [ ]:
previous_trip=df2.groupBy('individual_id', 'travel_type_description_previous').count()

previous_trip1=previous_trip.groupBy('individual_id').pivot('travel_type_description_previous').agg(sum('count')).fillna(0)

previous_trip1.show()

In [ ]:
previous_trip2=U.add_table_name(previous_trip1, 'previous_trip', exclude=['individual_id'])

previous_trip3=U.columnsTransform(previous_trip2)

previous_trip3.show()

## Extract Insurance purchased as a feature

In [ ]:
df.groupBy('insurancepurchased').count().show(20, False)

In [ ]:
insurance_list=['VENDOR', 'ACCEPTED', 'AAA TRIP PROTECTION AND VENDOR', 'INSURANCE PURCHASED INDEPENDENTLY','AAA TRIP PROTECTION ON SEPARATE INVOICE', 'AAA TRIP PROTECTION', 'BOTH']

# Standardize to uppercase and trim
df1 = df.withColumn("insurance_clean", trim(upper(col("insurancepurchased"))))

# Also convert list values to uppercase
insurance_list_upper = [x.upper() for x in insurance_list]

# Apply the condition
df1 = df1.withColumn(
    'insurance_purchased_index',
    when(col('insurance_clean').isin(insurance_list_upper), lit(1)).otherwise(lit(0))
)

df1.groupBy('insurance_purchased_index').count().show(20, False)

In [ ]:
df1.groupBy('individual_id', 'insurance_purchased_index').count().show()

In [ ]:
insurance_purchased=df1.filter(col('travel_type_description')=='Ship').groupBy('individual_id', 'insurance_purchased_index').count()

insurance_purchased1=insurance_purchased.groupBy('individual_id').pivot('insurance_purchased_index').agg(sum('count')).fillna(0)

insurance_purchased2=insurance_purchased1.withColumn('percent_of_insurance_purchased', col('1')/((col('0')+col('1'))))

insurance_purchased2.show()

In [ ]:
insurance_purchased3=insurance_purchased2.select('individual_id', 'percent_of_insurance_purchased')

In [ ]:
insurance_purchased3.count()

# add both features to existing file

In [ ]:
existing_features=spark.read.parquet(data_path+'transformed_features_cruiser_cancellation.parquet')

In [ ]:
existing_features.count()

In [ ]:
# join previous trip feature
# need to use left, since some people may only book one trip, therefore, there is no previous trip.

combined=existing_features.join(previous_trip3, 'individual_id', 'left').fillna(0)
combined.count()

In [ ]:
# join insurance purchased feature
combined1=combined.join(insurance_purchased3, 'individual_id', 'inner')

combined1.count()

In [ ]:
combined1.write.mode('overwrite').parquet(data_path+'transformed_features_cruiser_cancellation_v2.parquet')